In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_18_long_range_potentials_and_ewald.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 18: Neighbor Lists, Long-Range Potentials, and Ewald Summation

_Session d1769172 · Estimated time: ~135 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Reduce short-range pair force complexity from O(N^2) to O(N) using Verlet neighbor lists with skin depth Δ and O(N)-memory linked-cell lists.
- Derive analytic long-range energy and virial pressure tail corrections for short-range potentials (ν > D) and apply the mathematical definition of a long-range potential (ν ≤ D).
- Explain the Stillinger–Lovett screening condition S_qq(k) = c k^2 at low k for charged fluids and why naive spherical truncation of Coulomb forces fails catastrophically.
- Evaluate the conditionally convergent 1D alternating ionic chain Madelung constant α_1D = 2 ln 2 ≈ 1.38629 and compare 3D Madelung constants (NaCl, CsCl, ZnS).
- Decompose the periodic Coulomb interaction via the Ewald summation method into real-space erfc(κr)/r, reciprocal-space Gaussian exp(-k^2/4κ^2)/k^2, self-interaction -κ/√π ∑ q_i^2, and macroscopic surface dipole terms.
- Factor the reciprocal-space double sum into single-particle structure factors ρ_k using recursive complex multiplications, and compare theNatoli–Ceperley optimized breakup against classic Gaussian Ewald.
- Derive the optimal Ewald complexity O(N^(3/2)) and compare it with Particle-Mesh Ewald (PME/PPPM, O(N ln N)) and the Fast Multipole Method (FMM, O(N)).


In [2]:
ws.render_section('sec_pbc_tail_corrections_and_neighbor_lists')

### Periodic Distances and the Minimum Image Convention
In a periodic simulation box of side length $L$, the **Minimum Image Convention (MIC)** computes the interaction between particles $i$ and $j$ using only the closest periodic image:

$$|\mathbf{r}_{ij}|_M = \min_{\mathbf{n} \in \mathbb{Z}^3} |\mathbf{r}_i - \mathbf{r}_j + \mathbf{n}L|$$

For the pair potential and force to remain continuous as particles cross the mid-planes of the box, a spherically symmetric potential without image summation **must be truncated at a cutoff radius $r_c \le L/2$** ($V(r) = 0$ for $r > r_c$).

### Analytic Tail Corrections for Short-Range Potentials ($\nu > D$)
When we truncate a short-ranged potential $V(r)$ at $r_c$, we can restore the missing long-range attraction via thermodynamic perturbation theory by assuming that the radial distribution function approaches the uniform ideal-gas limit **$g(r) \approx 1$ for $r > r_c$**:

* **Energy Tail Correction per Particle (3D):**
 $$\frac{\Delta E_{\text{tail}}}{N} = \frac{\rho}{2} \int_{r_c}^{\infty} d^3r\, g(r) V(r) \approx 2\pi\rho \int_{r_c}^{\infty} r^2 V(r)\,dr$$
* **Virial Pressure Tail Correction (3D):**
 $$\Delta P_{\text{tail}} = -\frac{2\pi\rho^2}{3} \int_{r_c}^{\infty} r^3 \frac{dV(r)}{dr}\,dr$$

For the **Lennard-Jones 12-6 potential** $V_{\text{LJ}}(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6\right]$, integrating from $r_c$ to $\infty$ yields closed-form corrections:

$$\frac{\Delta E_{\text{tail}}^{\text{LJ}}}{N} = \frac{8\pi\rho\varepsilon\sigma^3}{3}\left[ \frac{1}{3}\left(\frac{\sigma}{r_c}\right)^9 - \left(\frac{\sigma}{r_c}\right)^3 \right], \qquad \Delta P_{\text{tail}}^{\text{LJ}} = \frac{16\pi\rho^2\varepsilon\sigma^3}{3}\left[ \frac{2}{3}\left(\frac{\sigma}{r_c}\right)^9 - \left(\frac{\sigma}{r_c}\right)^3 \right]$$

*(If truncating the potential significantly perturbs $g(r)$ itself, one must solve an integral equation such as the **RHNC — Reference Hypernetted-Chain** equation).*

---

### Reducing Force Complexity from $\mathcal{O}(N^2)$ to $\mathcal{O}(N)$: Neighbor Lists & Cell Lists
Evaluating all $N(N-1)/2$ pairs directly scales as $\mathcal{O}(N^2)$. However, if $V(r) = 0$ for $r > r_c$, each particle interacts only with an average of $m = \frac{4}{3}\pi\rho r_c^3$ neighbors inside $r_c$, where **$m$ is independent of system size $N$**!

1. **Verlet Neighbor List with Skin Depth $\Delta$:**
 * For each particle, we store a table of all neighbors within an extended radius **$r_{\text{list}} = r_c + \Delta$** (where $\Delta$ is the **skin depth**).
 * Force evaluation checks only the table entries: $\mathcal{O}(m N) = \mathcal{O}(N)$ per step.
 * We track the **two largest scalar displacements $d_1$ and $d_2$** traveled by any particles since the last table build. As soon as **$d_1 + d_2 \ge \Delta$**, a particle outside the list could have crossed inside $r_c$, so we rebuild the neighbor table.
 * **Tradeoff in $\Delta$:** As $\Delta$ decreases, $m \propto (r_c + \Delta)^3$ shrinks (fewer force evaluations per step), but the table must be rebuilt more frequently. As $\Delta$ increases, rebuilds are rarer, but each step checks more pairs. Modern MD codes dynamically optimize $\Delta$ by fitting the CPU time $t(\Delta)$ to a polynomial and minimizing it.
2. **The Linked-Cell (Bin Sort) Method:**
 * Naive Verlet table construction checks all pairs and costs $\mathcal{O}(N^2)$ (and requires $\mathcal{O}(mN)$ memory).
 * Dividing the simulation box into a regular mesh of cells of side length **$L_{\text{cell}} > r_c + \Delta$** and sorting particles into a **linked list** (head-of-chain + next-particle arrays) reduces **table construction to $\mathcal{O}(N)$** (checking only a particle's own cell and its $3^3 - 1 = 26$ adjacent cells in 3D) and reduces **memory to $\mathcal{O}(N)$**!


In [3]:
ws.render_section('concept_neighbor_skin')

### Interactive Explorer: Verlet Neighbor List Skin Depth Δ Optimization

Adjust the number of particles $N$, potential cutoff $r_c$, and table rebuild algorithm ($\mathcal{O}(N^2)$ all-pairs vs. $\mathcal{O}(N)$ linked-cell) to see how the optimal skin depth $\Delta^*$ balances per-step pair evaluations against amortized table rebuild cost.

**Key Takeaway:** Choosing $\Delta$ too small causes frequent table rebuilds; choosing $\Delta$ too large bloats the sphere volume $\propto (r_c + \Delta)^3$. Combining a linked-cell mesh with a Verlet skin list makes both force evaluation and table construction strictly $\mathcal{O}(N)$.

In [4]:
ws.render_section('q01_minimum_image_cutoff_limit')

**Q1 · easy** · 💡

Under the Minimum Image Convention in a cubic periodic box of side length $L$, why must a spherically truncated short-range potential satisfy $V(r) = 0$ for $r > L/2$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what happens to the force on a particle as a neighbor crosses the midplane $x_j - x_i = \pm L/2$.

</div>
</details>

- [ ] **(A)** So that the potential and force remain continuous across the minimum-image boundary at $r = L/2$ and avoid self-overlapping images.
- [ ] **(B)** Because the radial distribution function $g(r)$ is identically zero for $r > L/2$ in all liquids.
- [ ] **(C)** Because the Fourier transform of any function vanishes for wavelengths larger than $L/2$.
- [ ] **(D)** Because computer floating-point numbers cannot represent distances larger than $L/2$.

In [5]:
ws.render_section('q02_verlet_skin_rebuild_trigger')

**Q2 · easy** · 💡

In a Verlet neighbor list cut off at $r_c + \Delta$ (where $\Delta$ is the skin depth), we monitor the two largest distances $d_1$ and $d_2$ traveled by any particles since the table was last constructed. What exact condition triggers a neighbor-table rebuild?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the worst case where two particles initially just outside $r_c + \Delta$ move directly toward each other.

</div>
</details>

- [ ] **(A)** When $d_1 + d_2 \ge \Delta$
- [ ] **(B)** When $d_1 - d_2 \ge r_c$
- [ ] **(C)** When $d_1^2 + d_2^2 \ge L/2$
- [ ] **(D)** Only after a fixed number of 10,000 steps regardless of particle velocities

In [6]:
ws.render_section('q03_cell_method_complexity_and_memory')

**Q3 · medium** · 💡

How does the linked-cell (bin sort) method improve upon a pure all-pairs Verlet neighbor table for $N$ particles with average neighbor count $m$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare both the table-construction CPU scaling and the array memory scaling.

</div>
</details>

- [ ] **(A)** It reduces neighbor-table construction from $\mathcal{O}(N^2)$ to $\mathcal{O}(N)$ and reduces linked-list memory storage from $\mathcal{O}(mN)$ to $\mathcal{O}(N)$.
- [ ] **(B)** It eliminates the need for periodic boundary conditions and allows $r_c > 2L$.
- [ ] **(C)** It computes exact long-range $1/r$ Coulomb forces in $\mathcal{O}(1)$ time without Fourier transforms.
- [ ] **(D)** It increases memory usage to $\mathcal{O}(N^3)$ in exchange for zero force evaluations.

In [7]:
ws.render_section('q04_lj_energy_tail_correction_numeric')

**Q4 · medium** · 💡

Consider an attractive dispersion tail $V(r) = -4\varepsilon(\sigma/r)^6$ truncated at $r_c = 2.0\sigma$ in a 3D fluid of number density $\rho\sigma^3 = 0.75$ with $\varepsilon = 1.0$. Using the perturbation tail formula with $g(r) = 1$ for $r > r_c$:

$$\frac{\Delta E_{\text{tail}}}{N} = 2\pi\rho \int_{r_c}^{\infty} r^2 V(r)\,dr = -\frac{8\pi\rho\varepsilon\sigma^3}{3}\left(\frac{\sigma}{r_c}\right)^3$$

calculate $\Delta E_{\text{tail}}/N$ (in units of $\varepsilon$, rounded to 3 decimal places).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Integrate the attractive $r^{-6}$ tail beyond $r_c$ assuming uniform radial distribution $g(r) \approx 1$.

</div>
</details>

_Numeric answer required._

In [8]:
ws.render_section('q05_lj_pressure_tail_exponent')

**Q5 · medium** · 💡

For a pair potential with an attractive tail $V(r) = -C_6 r^{-6}$ beyond $r_c$ in 3D, how does the virial pressure tail correction $\Delta P_{\text{tail}} = -\frac{2\pi\rho^2}{3}\int_{r_c}^\infty r^3 \frac{dV}{dr}\,dr$ scale with the number density $\rho$ and the cutoff radius $r_c$, and what is its sign?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Note that $dV/dr = +6 C_6 r^{-7} > 0$, so $r^3 (dV/dr) = 6 C_6 r^{-4}$.

</div>
</details>

- [ ] **(A)** $\Delta P_{\text{tail}}$ is negative (attractive tail lowers the pressure) and scales as $-\rho^2 / r_c^3$.
- [ ] **(B)** $\Delta P_{\text{tail}}$ is positive and scales as $+\rho / r_c^6$.
- [ ] **(C)** $\Delta P_{\text{tail}}$ is negative and scales as $-\rho^3 / r_c^2$.
- [ ] **(D)** $\Delta P_{\text{tail}}$ is identically zero because $dV/dr$ integrates to zero.

In [9]:
ws.render_section('sec_long_range_definition_and_stillinger_lovett')

### What Makes a Potential "Long-Ranged"?
A pair potential $V(r) \sim r^{-\nu}$ is mathematically defined as **long-ranged** if its spatial tail integral (or real-space lattice sum) diverges at infinity. In **$D$ spatial dimensions**, the volume of a spherical shell of radius $r$ grows as $r^{D-1}\,dr$, so the tail integral is:

$$\int_{r_c}^{\infty} d^D r\, V(r) \propto \int_{r_c}^{\infty} r^{D-1} r^{-\nu}\,dr = \int_{r_c}^{\infty} r^{D - 1 - \nu}\,dr$$

This integral **diverges whenever $\nu \le D$**:
* **In 3D ($D = 3$):** A potential is long-ranged if it decays at a rate **$\nu \le 3$** (i.e., slower than or equal to $r^{-3}$). Thus **Coulomb interactions ($1/r$, $\nu=1$)**, **charge-dipole interactions ($1/r^2$, $\nu=2$)**, **dipole-dipole interactions ($1/r^3$, $\nu=3$)**, and **gravitation ($1/r$, $\nu=1$)** are all strictly long-ranged in 3D, whereas dispersion ($1/r^6$, $\nu=6 > 3$) is short-ranged!
* **In 2D ($D = 2$):** A potential is long-ranged if it decays at a rate **$\nu \le 2$**.

---

### Why Naive Cutoff Fails for Charged Systems: The Stillinger–Lovett Condition
What goes wrong if we treat a charged ionic fluid like a short-range fluid and simply truncate $V(r) = q_i q_j / r$ at $r_c = L/2$?
1. **Growing boundary discontinuity:** Because the number of ions in a shell at distance $r$ grows as $4\pi\rho r^2\,dr$, the contribution of the cutoff jump $(1/r)(r^2) \sim r$ **grows larger and larger with box size $L$** instead of disappearing!
2. **Violation of Poisson's Equation and the Stillinger–Lovett Screening Condition:**
 In an uncharged (neutral) fluid, long-wavelength density fluctuations are governed by the isothermal compressibility:
 $$\lim_{k\to 0} S(k) = c = \rho k_B T \chi_T > 0 \quad (\text{uncharged system})$$
 In contrast, in a **charged conducting/ionic system**, long-range $1/r$ Coulomb forces (which transform as $4\pi/k^2$ in Fourier space via Poisson's equation $\nabla^2\phi = -4\pi\rho_q$) enforce **complete macroscopic charge screening**. The **Stillinger–Lovett second-moment condition** requires the charge-charge structure factor $S_{qq}(k)$ to vanish quadratically at small $k$:
 $$S_{qq}(k) = c\,k^2 + \mathcal{O}(k^4) \quad \text{as } k \to 0 \quad (\text{charged system})$$
 Truncating $1/r$ at $r_c$ breaks Poisson's equation at long wavelengths ($k < 2\pi/r_c$), violating the Stillinger–Lovett condition and creating unphysical artificial charge layers at $r = r_c$.

---

### Conditional Convergence and the 1D Madelung Constant
Consider a one-component lattice of $+1$ charges: $\sum_{\mathbf{L}\ne\mathbf{0}} \frac{1}{|\mathbf{r}_i - \mathbf{L}|} \approx \int_0^\infty 4\pi r^2 \frac{\rho}{r}\,dr = \infty$. A non-neutral infinite periodic system has **infinite electrostatic energy** unless neutralized by a uniform background (jellium) or opposite ions ($\sum_i q_i = 0$).

Even in a strictly **charge-neutral ionic crystal** ($\pm q$ alternating), the Coulomb lattice sum is **conditionally convergent**—its value depends on the order in which terms are summed!

#### Exact Derivation of the 1D Alternating Chain Madelung Constant
Consider an infinite 1D chain of alternating $+q$ and $-q$ ions spaced by nearest-neighbor distance $R$ ($r_{ij} = R p_{ij}$). Starting on a reference negative ion at the origin and summing symmetrically over shells to the left and right ($2$ neighbors at distance $p_{ij} = 1, 2, 3, \dots$):

$$V_{\text{electrostatic}} = -\frac{\alpha q^2}{R}, \qquad \alpha_{\text{1D}} = \sum_{j\ne i} \frac{(\pm 1)}{p_{ij}} = 2\left[ \frac{1}{1} - \frac{1}{2} + \frac{1}{3} - \frac{1}{4} + \cdots \right]$$

Using the Maclaurin series $\ln(1 + x) = x - \frac{x^2}{2} + \frac{x^3}{3} - \frac{x^4}{4} + \cdots$ evaluated at $x = 1$:

$$\alpha_{\text{1D}} = 2 \ln(1 + 1) = 2 \ln 2 \approx 1.386294$$

#### 3D Madelung Constants
In 3D, naive spherical shell sums do not even converge unless successive shells are grouped into fractional neutral units (Evjen 1932; Frank 1950) or resummed in Fourier space (Ewald 1921):
* **Rock Salt ($\text{NaCl}$, 6-coordinated):** $\alpha_{\text{NaCl}} = 1.747565$
* **Cesium Chloride ($\text{CsCl}$, 8-coordinated):** $\alpha_{\text{CsCl}} = 1.762675$
* **Zincblende ($\text{ZnS}$, 4-coordinated):** $\alpha_{\text{ZnS}} = 1.6381$


In [10]:
ws.render_section('concept_madelung_1d')

### Interactive Explorer: 1D Madelung Sum — Conditional Direct Convergence vs. Ewald

Compare the slow, oscillatory convergence of the direct alternating sum $2\sum_{n=1}^M (-1)^{n-1}/n$ against the rapid exponential convergence of the Ewald sum toward $\alpha_{\text{1D}} = 2\ln 2 \approx 1.386294$.

**Key Takeaway:** The direct alternating Coulomb sum $2\sum (-1)^{n-1}/n$ is only conditionally convergent, oscillating with $\mathcal{O}(1/M)$ error, whereas Ewald summation converts it into two exponentially convergent series that reach 6-digit accuracy within a few shells.

In [11]:
ws.render_section('q06_long_range_dimension_criterion')

**Q6 · easy** · 💡

In $D$ spatial dimensions, a pair potential $V(r) \sim r^{-\nu}$ is mathematically classified as **long-ranged** if its real-space tail integral $\int_{r_c}^\infty r^{D-1} r^{-\nu}\,dr$ diverges. What are the critical decay exponents $\nu$ in 3D and 2D?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check when $\int_{r_c}^\infty r^{D - 1 - \nu}\,dr$ fails to converge at the upper limit.

</div>
</details>

- [ ] **(A)** Long-ranged if $\nu \le 3$ in 3D, and if $\nu \le 2$ in 2D.
- [ ] **(B)** Long-ranged if $\nu \le 6$ in 3D, and if $\nu \le 4$ in 2D.
- [ ] **(C)** Long-ranged only if $\nu < 1$ in all dimensions.
- [ ] **(D)** Long-ranged if $\nu > 3$ in 3D, and if $\nu > 2$ in 2D.

In [12]:
ws.render_section('q07_which_potentials_are_long_ranged_3d')

**Q7 · medium** · 💡

According to the 3D long-range criterion ($\nu \le 3$), which of the following interactions are strictly **long-ranged** in three dimensions? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the exponent $\nu$ in $r^{-\nu}$ for each interaction against $\nu \le 3$.

</div>
</details>

- [ ] **(A)** Ion–ion Coulomb potential $V(r) \propto 1/r$ ($\nu = 1$)
- [ ] **(B)** Newtonian gravitational potential $V(r) \propto -1/r$ ($\nu = 1$)
- [ ] **(C)** Unaveraged dipole–dipole interaction $V(r) \propto 1/r^3$ ($\nu = 3$)
- [ ] **(D)** London dispersion attraction $V(r) \propto -1/r^6$ ($\nu = 6$)
- [ ] **(E)** Lennard-Jones Pauli repulsion $V(r) \propto 1/r^{12}$ ($\nu = 12$)

In [13]:
ws.render_section('q08_stillinger_lovett_condition')

**Q8 · medium** · 💡

How does the small-wavevector ($k \to 0$) behavior of the structure factor $S(k)$ in a **charged ionic system** differ from that of an **uncharged neutral fluid** according to the Stillinger–Lovett screening condition?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Perturbation theory' for $S(k)$ at low $k$ in charged vs. Uncharged systems.

</div>
</details>

- [ ] **(A)** For a charged system $S(k) = c k^2$ as $k \to 0$ (perfect screening of long-wavelength charge fluctuations), whereas for an uncharged system $S(k) \to c > 0$.
- [ ] **(B)** For a charged system $S(k) \to \infty$ as $1/k^2$, whereas for an uncharged system $S(k) = 0$ identically for all $k$.
- [ ] **(C)** For both charged and uncharged systems, $S(k) = 1$ at $k = 0$.
- [ ] **(D)** For a charged system $S(k) = c k^4$, whereas for an uncharged system $S(k) = c k^2$.

In [14]:
ws.render_section('q09_why_cutoff_coulomb_fails')

**Q9 · medium** · 💡

Why is it physically unacceptable to handle a charged system simply by truncating the Coulomb potential $1/r$ at $r_c = L/2$ like a short-range potential?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Charged systems' for the two key problems with a cutoff Coulomb potential.

</div>
</details>

- [ ] **(A)** The boundary discontinuity effect $(1/r)(r^2) \sim r$ grows with cutoff radius instead of vanishing, and truncation violates Poisson's equation and the Stillinger–Lovett screening condition.
- [ ] **(B)** The Coulomb potential is zero for $r < L/2$ and only turns on for $r > L/2$.
- [ ] **(C)** Truncating 1/r causes the particle masses to become negative.
- [ ] **(D)** Verlet neighbor lists cannot store pairs when charges have opposite signs.

In [15]:
ws.render_section('q10_madelung_1d_exact_value_numeric')

**Q10 · easy** · 💡

Calculate the exact Madelung constant $\alpha_{\text{1D}} = 2\sum_{n=1}^{\infty} \frac{(-1)^{n-1}}{n}$ for an infinite 1D chain of alternating $\pm q$ ions with nearest-neighbor spacing $R$ (round to 4 decimal places).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recognize the alternating harmonic series expansion of the natural logarithm of 2.

</div>
</details>

_Numeric answer required._

In [16]:
ws.render_section('q11_madelung_1d_partial_sum_numeric')

**Q11 · easy** · 💡

In the 1D alternating ionic chain, if you truncate the direct sum after just $M = 3$ neighbor shells on each side:

$$\alpha_3 = 2\left(1 - \frac{1}{2} + \frac{1}{3}\right)$$

what is the value of $\alpha_3$ (round to 3 decimal places)? Notice how far it still is from $2\ln 2 \approx 1.386$!

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the first three terms of the alternating series and multiply by the two-sided line prefactor.

</div>
</details>

_Numeric answer required._

In [17]:
ws.render_section('q12_madelung_3d_coordination_ordering')

**Q12 · medium** · 💡

Rank the three cubic ionic crystal structures—Zincblende ($\text{ZnS}$, 4-coordinated), Rock Salt ($\text{NaCl}$, 6-coordinated), and Cesium Chloride ($\text{CsCl}$, 8-coordinated)—in order of **increasing** 3D Madelung constant $\alpha$ (defined relative to nearest-neighbor distance $R$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the table: ZnS (1.6381), NaCl (1.747565), CsCl (1.762675).

</div>
</details>

- [ ] **(A)** $\alpha(\text{ZnS}) = 1.6381 < \alpha(\text{NaCl}) = 1.747565 < \alpha(\text{CsCl}) = 1.762675$
- [ ] **(B)** $\alpha(\text{CsCl}) < \alpha(\text{NaCl}) < \alpha(\text{ZnS})$
- [ ] **(C)** $\alpha(\text{NaCl}) < \alpha(\text{ZnS}) < \alpha(\text{CsCl})$
- [ ] **(D)** All three 3D structures have the exact same Madelung constant $\alpha = 2\ln 2$.

In [18]:
ws.render_section('q13_conditional_convergence_meaning')

**Q13 · medium** · 💡

What does it mean mathematically and physically that the electrostatic lattice sum $\sum_{j\ne i} q_i q_j / r_{ij}$ in a neutral crystal is **conditionally convergent**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the definition of conditional convergence for an infinite series whose absolute values diverge ($\sum 1/r = \infty$).

</div>
</details>

- [ ] **(A)** The sum of absolute values diverges ($\sum |q_i q_j|/r_{ij} = \infty$), so the series only converges if positive and negative terms are grouped to nearly cancel, and its limiting value depends on the summation order / macroscopic surface shape and boundary conditions.
- [ ] **(B)** The sum converges only when the temperature $T$ is below the Debye temperature.
- [ ] **(C)** Every rearrangement of the terms in the series converges to the exact same number.
- [ ] **(D)** The sum is finite only when all charges have the same sign ($+q$).

In [19]:
ws.render_section('sec_ewald_summation_method')

### Origin of the Convergence Problem in $r$-Space vs. $k$-Space
For a periodic function $f(\mathbf{r} + \mathbf{L}_n) = f(\mathbf{r})$ in a box of volume $\Omega = L_x L_y L_z$, the discrete Fourier expansion uses reciprocal lattice vectors $\mathbf{k} = 2\pi\left(\frac{m_x}{L_x}\hat{\mathbf{x}} + \frac{m_y}{L_y}\hat{\mathbf{y}} + \frac{m_z}{L_z}\hat{\mathbf{z}}\right)$ with $m_x, m_y, m_z \in \mathbb{Z}$:

$$v(\mathbf{r}) = \frac{e^2}{r} \Longleftrightarrow v_{\mathbf{k}} = \frac{1}{\Omega}\int d^3r\, e^{-i\mathbf{k}\cdot\mathbf{r}} \frac{e^2}{r} = \frac{4\pi e^2}{\Omega k^2}$$

Why can't we just do the entire sum in real space *or* the entire sum in $k$-space?
* **$r$-space convergence issue comes from $r \to \infty$ ($k \to 0$):** $1/r$ has a long tail at large $r$.
* **$k$-space convergence issue comes from $k \to \infty$ ($r \to 0$):** Because $1/r$ has a sharp $r \to 0$ singularity in real space, its Fourier transform $4\pi e^2 / k^2$ decays slowly at large $k$!

---

### Ewald's Physical Trick: Screening with Neutralizing Gaussian Clouds
Because **Poisson's equation is linear**, we can split the Coulomb potential into a rapidly decaying **short-range real-space part** and a smooth **long-range reciprocal-space part** by surrounding every point charge $q_i$ with a diffuse, neutralizing **Gaussian charge distribution** of width $1/\kappa$:

$$\rho_i^{\text{Gauss}}(\mathbf{r}) = -q_i \left(\frac{\kappa^2}{\pi}\right)^{3/2} e^{-\kappa^2 |\mathbf{r} - \mathbf{r}_i|^2}$$

and then **adding back** the exact same compensating Gaussian charge distribution $+\rho_i^{\text{Gauss}}(\mathbf{r})$ to be summed in $k$-space! Using the error function $\text{erf}(z) = \frac{2}{\sqrt{\pi}}\int_0^z e^{-t^2}\,dt$ and its complement $\text{erfc}(z) = 1 - \text{erf}(z)$, the identity $\frac{1}{r} = \frac{\text{erfc}(\kappa r)}{r} + \frac{\text{erf}(\kappa r)}{r}$ splits the pair interaction into:

1. **Short-Range Real-Space Potential ($v_{\text{short}}$):**
 $$v_{\text{short}}(r) = \frac{q_1 q_2}{r}\text{erfc}(\kappa r) \qquad (\text{decays exponentially fast as } e^{-\kappa^2 r^2}\text{ at large } r)$$
 We choose the **Ewald parameter $\kappa$** large enough that $v_{\text{short}}(L/2) = \frac{2q_1 q_2}{L}\text{erfc}(\kappa L/2) \approx 0$ at the minimum-image boundary $r = L/2$, so we need **only the minimum image** in real space!
2. **Long-Range Reciprocal-Space Potential ($v_{\text{long}}$):**
 $$v_{\text{long}}(r) = \frac{q_1 q_2}{r}\text{erf}(\kappa r) \Longleftrightarrow v_{\text{long},\mathbf{k}} = \frac{4\pi q_1 q_2}{\Omega k^2} e^{-k^2 / (4\kappa^2)} \qquad (\text{decays exponentially fast at large } k)$$
 Because $\text{erf}(\kappa r)/r \to \frac{2\kappa}{\sqrt{\pi}}$ is completely smooth and non-singular as $r \to 0$, its Fourier transform acquires the Gaussian damping factor $e^{-k^2/(4\kappa^2)}$, allowing truncation at a modest wavevector cutoff $|\mathbf{k}| < k_c$!

---

### The Four Complete Ewald Energy Terms
Combining all pairs and periodic images in a charge-neutral cell ($\sum_i q_i = 0$ removes the $\mathbf{k} = \mathbf{0}$ divergence) yields the complete Ewald electrostatic energy:

$$V_{\text{Ewald}} = V_{\text{real}} + V_{\text{recip}} + V_{\text{self}} + V_{\text{dipole}}$$

* **1. Real-Space Term:**
 $$V_{\text{real}} = \frac{1}{2} \sum_{i \ne j}^N \frac{q_i q_j}{r_{ij}} \text{erfc}(\kappa r_{ij}), \qquad r_{ij} = \min_{\mathbf{L}} |\mathbf{r}_i - \mathbf{r}_j - \mathbf{L}| < r_c$$
* **2. Reciprocal-Space Term:**
 $$V_{\text{recip}} = \frac{1}{2\Omega} \sum_{\mathbf{k} \ne \mathbf{0}, |\mathbf{k}| < k_c} \frac{4\pi}{k^2} e^{-k^2/(4\kappa^2)} \left| \sum_{j=1}^N q_j e^{i\mathbf{k}\cdot\mathbf{r}_j} \right|^2$$
* **3. Self-Interaction Subtraction Term ($C = V_{\text{self}}$):**
 When we write the unrestricted double sum $\frac{1}{2}\sum_{i,j} q_i q_j e^{i\mathbf{k}\cdot(\mathbf{r}_i - \mathbf{r}_j)} v_{\text{long},\mathbf{k}} = \frac{1}{2}\sum_{\mathbf{k}} |\rho_{\mathbf{k}}|^2 v_{\mathbf{k}}$ in $k$-space, we inadvertently include the $i = j$ self-interaction of each point charge with its own compensating Gaussian cloud at $r = 0$. Since $\lim_{r\to 0} \frac{\text{erf}(\kappa r)}{r} = \frac{2\kappa}{\sqrt{\pi}}$, we must subtract this constant self-energy:
 $$V_{\text{self}} = -\frac{1}{2}\sum_{i=1}^N q_i^2 \lim_{r\to 0} \frac{\text{erf}(\kappa r)}{r} = -\frac{\kappa}{\sqrt{\pi}} \sum_{i=1}^N q_i^2$$
* **4. Macroscopic Surface Dipole Term ($V_{\text{dipole}}$ for Insulators):**
 For a macroscopic sphere of periodic replicas embedded in an external medium of dielectric constant $\varepsilon_{\text{ext}}$, surface polarization adds a depolarization energy proportional to the square of the total simulation cell dipole moment $\mathbf{M} = \sum_i \boldsymbol{\mu}_i = \sum_{i=1}^N q_i \mathbf{r}_i$:
 $$V_{\text{dipole}} = \frac{2\pi}{(2\varepsilon_{\text{ext}} + 1)\Omega} \left| \sum_{i=1}^N q_i \mathbf{r}_i \right|^2$$
 * **Tin-foil (conducting metal) boundary conditions ($\varepsilon_{\text{ext}} \to \infty$):** Surface charges on the surrounding conductor screen out the macroscopic dipole field completely, so **$V_{\text{dipole}} = 0$**.
 * **Vacuum boundary conditions ($\varepsilon_{\text{ext}} = 1$):** $V_{\text{dipole}} = \frac{2\pi}{3\Omega}\left|\sum_i q_i \mathbf{r}_i\right|^2$.


In [20]:
ws.render_section('concept_ewald_splitting')

### Interactive Explorer: Ewald Screening Parameter κ — Real Space, k-Space, and CPU Tradeoff

Vary the Ewald screening parameter $\kappa$ and switch between the real-space potential split ($v_{\text{short}}$ vs. $v_{\text{long}}$), the $k$-space Gaussian envelope $v_{\text{long},k}$, and the total CPU operation count.

**Key Takeaway:** Increasing $\kappa$ makes the Gaussian screening cloud narrower in real space (so $\text{erfc}(\kappa r)/r$ decays faster and needs a smaller real-space cutoff $r_c$), but wider in $k$-space (so $e^{-k^2/(4\kappa^2)}$ decays slower and requires more $\mathbf{k}$-vectors).

In [21]:
ws.render_section('q14_convergence_origin_r_vs_k')

**Q14 · easy** · 💡

For the bare Coulomb potential $v(r) = e^2/r$ with Fourier transform $v_k = 4\pi e^2/(\Omega k^2)$, where do the slow convergence problems originate in real space ($r$-space) versus reciprocal space ($k$-space)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Origin of Convergence Problem'.

</div>
</details>

- [ ] **(A)** The real-space convergence issue comes from $r \to \infty$ ($k \to 0$), whereas the reciprocal-space convergence issue comes from $k \to \infty$ (the sharp $1/r$ singularity at $r \to 0$).
- [ ] **(B)** Both the real-space and reciprocal-space convergence issues come solely from $r \to \infty$.
- [ ] **(C)** The real-space convergence issue comes from $r \to 0$, whereas the reciprocal-space convergence issue comes from $k \to 0$.
- [ ] **(D)** Neither r-space nor k-space has any convergence issue in 3D.

In [22]:
ws.render_section('q15_ewald_gaussian_screening_idea')

**Q15 · easy** · 💡

What is the physical 'trick' behind the classic Ewald summation breakup of the Coulomb potential?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Classic Ewald': why are we free to add and subtract Gaussian charge clouds?

</div>
</details>

- [ ] **(A)** Because Poisson's equation is linear, we add a neutralizing Gaussian charge cloud around each point charge to screen out the long-range part in real space, and subtract the same Gaussian cloud in smooth k-space.
- [ ] **(B)** We replace all point charges with neutral hard spheres of diameter $L/2$.
- [ ] **(C)** We set the permittivity of free space to infinity inside a radius $r_c$ and zero outside $r_c$.
- [ ] **(D)** We multiply all ionic charges by $\exp(-t/\tau)$ so that the system becomes uncharged at long times.

In [23]:
ws.render_section('q16_ewald_short_and_long_formulas')

**Q16 · medium** · 💡

In the classic Gaussian Ewald split $v(r) = v_{\text{short}}(r) + v_{\text{long}}(r)$, what are the exact functional forms of $v_{\text{short}}(r)$ in real space and $v_{\text{long},\mathbf{k}}$ in reciprocal space?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Classic Ewald' for $v_{\text{short}}(r)$ and $v_{\text{long},k}$.

</div>
</details>

- [ ] **(A)** $v_{\text{short}}(r) = \frac{q_1 q_2}{r} \operatorname{erfc}(\kappa r)$ and $v_{\text{long}}(k) = \frac{4\pi q_1 q_2}{\Omega k^2} \exp\left(-\frac{k^2}{4\kappa^2}\right)$
- [ ] **(B)** $v_{\text{short}}(r) = \frac{q_1 q_2}{r} \operatorname{erf}(\kappa r)$ and $v_{\text{long}}(k) = \frac{4\pi q_1 q_2}{\Omega k^2} \exp(-\kappa^2 k^2)$
- [ ] **(C)** $v_{\text{short}}(r) = \frac{q_1 q_2}{r^2} \exp(-\kappa r)$ and $v_{\text{long}}(k) = \frac{4\pi q_1 q_2}{\Omega k^4}$
- [ ] **(D)** $v_{\text{short}}(r) = q_1 q_2 \operatorname{erfc}(r/\kappa)$ and $v_{\text{long}}(k) = \frac{q_1 q_2}{\Omega} \cos(k/\kappa)$

In [24]:
ws.render_section('q17_ewald_kappa_increase_effect')

**Q17 · medium** · 💡

If you **increase** the Ewald parameter $\kappa$ (making the real-space screening Gaussian narrower and taller) while holding target accuracy fixed, how do the required real-space cutoff $r_c$ and reciprocal-space wavevector cutoff $k_c$ change?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at $\text{erfc}(\kappa r_c)$ and $e^{-k_c^2/(4\kappa^2)}$ in the interactive explorer.

</div>
</details>

- [ ] **(A)** $v_{\text{short}}(r)$ decays faster so $r_c$ decreases (less real-space work), while $v_{\text{long}}(k)$ decays slower so $k_c$ increases (more $k$-space work).
- [ ] **(B)** Both $r_c$ and $k_c$ decrease simultaneously, reducing total work to $\mathcal{O}(1)$.
- [ ] **(C)** $v_{\text{short}}(r)$ decays slower so $r_c$ increases, while $v_{\text{long}}(k)$ decays faster so $k_c$ decreases.
- [ ] **(D)** Neither $r_c$ nor $k_c$ depends on $\kappa$.

In [25]:
ws.render_section('q18_ewald_self_energy_origin')

**Q18 · medium** · 💡

Why must the constant self-energy term $V_{\text{self}} = -\frac{\kappa}{\sqrt{\pi}}\sum_{i=1}^N q_i^2$ be subtracted from the reciprocal-space Ewald sum?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that the $k$-space structure factor product $|\rho_{\mathbf{k}}|^2 = \sum_{i,j} q_i q_j e^{i\mathbf{k}\cdot(\mathbf{r}_i - \mathbf{r}_j)}$ includes the diagonal $i = j$ terms.

</div>
</details>

- [ ] **(A)** Because factoring the $k$-space double sum into $|\rho_{\mathbf{k}}|^2$ includes the $i = j$ interaction of each point charge with its own compensating Gaussian cloud at $r = 0$, where $\lim_{r\to 0} \frac{\operatorname{erf}(\kappa r)}{r} = \frac{2\kappa}{\sqrt{\pi}}$.
- [ ] **(B)** Because $\frac{\operatorname{erfc}(\kappa r)}{r}$ approaches $\frac{2\kappa}{\sqrt{\pi}}$ at $r = \infty$.
- [ ] **(C)** Because the minimum image convention double-counts pairs at $r = L/2$.
- [ ] **(D)** Because the total charge $\sum q_i$ of an ionic crystal is always nonzero.

In [26]:
ws.render_section('q19_ewald_self_energy_numeric')

**Q19 · medium** · 💡

An Ewald simulation box contains $N = 100$ monovalent ions ($50$ cations with $q_i = +1$ and $50$ anions with $q_i = -1$, in electrostatic units where $4\pi\varepsilon_0 = 1$). If the Ewald screening parameter is set to $\kappa = 0.5\sqrt{\pi}\text{ Å}^{-1} \approx 0.8862\text{ Å}^{-1}$, what is the exact value of the Ewald self-energy correction $V_{\text{self}} = -\frac{\kappa}{\sqrt{\pi}}\sum_{i=1}^N q_i^2$ (in $\text{e}^2/\text{Å}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the self-energy formula using the ratio $\kappa / \sqrt{\pi}$ and the sum of squared charges.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q20_ewald_dipole_boundary_conditions')

**Q20 · medium** · 💡

In the Ewald summation for a macroscopically spherical assembly of periodic boxes embedded in an external medium of dielectric constant $\varepsilon$, the surface dipole correction is $V_{\text{dipole}} = \frac{2\pi}{(2\varepsilon + 1)\Omega}\left|\sum_i \boldsymbol{\mu}_i\right|^2$. How does $V_{\text{dipole}}$ behave for (i) a surrounding **conducting metal ('tin-foil' boundary, $\varepsilon \to \infty$)** versus (ii) a surrounding **vacuum (insulator in vacuum, $\varepsilon = 1$)**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $\frac{2\pi}{(2\varepsilon + 1)\Omega}$ as $\varepsilon \to \infty$ and at $\varepsilon = 1$.

</div>
</details>

- [ ] **(A)** For a conducting boundary ($\varepsilon \to \infty$), $V_{\text{dipole}} = 0$; for vacuum boundary conditions ($\varepsilon = 1$), $V_{\text{dipole}} = \frac{2\pi}{3\Omega} |\sum \boldsymbol{\mu}_i|^2$.
- [ ] **(B)** For a conducting boundary ($\varepsilon \to \infty$), $V_{\text{dipole}} \to \infty$; for vacuum ($\varepsilon = 1$), $V_{\text{dipole}} = 0$.
- [ ] **(C)** In both cases, $V_{\text{dipole}} = \frac{4\pi}{\Omega} |\sum \boldsymbol{\mu}_i|^2$.
- [ ] **(D)** $V_{\text{dipole}}$ is negative in vacuum and positive in a conductor.

In [28]:
ws.render_section('q21_ewald_dipole_term_numeric')

**Q21 · medium** · 💡

A cubic simulation box of volume $\Omega = 100\text{ nm}^3$ with **vacuum boundary conditions** ($\varepsilon = 1$) has an instantaneous total dipole moment of magnitude $|\sum_i \boldsymbol{\mu}_i| = 15.0$ (in reduced charge$\cdot\text{nm}$ units). Calculate the surface dipole energy $V_{\text{dipole}} = \frac{2\pi}{(2\varepsilon + 1)\Omega}\left|\sum_i \boldsymbol{\mu}_i\right|^2$ (round to 3 decimal places).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the prefactor for vacuum conditions ($\varepsilon = 1$) and scale by the square of the total dipole moment divided by box volume.

</div>
</details>

_Numeric answer required._

In [29]:
ws.render_section('q22_ewald_kspace_damping_numeric')

**Q22 · easy** · 💡

In reciprocal space, the Ewald long-range potential is damped by the Gaussian factor $e^{-k^2 / (4\kappa^2)}$. If $\kappa = 1.5\text{ nm}^{-1}$, what is the value of the Gaussian damping factor $e^{-k^2 / (4\kappa^2)}$ at wavevector magnitude $k = 3.0\text{ nm}^{-1}$ (round to 3 decimal places)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the dimensionless exponent using the given wavevector magnitude and screening parameter.

</div>
</details>

_Numeric answer required._

In [30]:
ws.render_section('sec_kspace_algorithm_natoli_and_scaling')

### Optimizing Breakup: The Natoli–Ceperley (1995) Method
In the conventional Gaussian Ewald breakup, truncating $v_{\text{short}}(r)$ at $r = L/2$ leaves a small residual jump $v_{\text{short}}(L/2) = \frac{2q_1 q_2}{L}\text{erfc}(\kappa L/2) \ne 0$, and the Gaussian is not the mathematically optimal splitting function for a given $(r_c, k_c)$ budget.

**Natoli and Ceperley (1995)** showed how to achieve machine-precision accuracy with minimal $k$-vectors by expanding the short-range (or long-range) potential in a set of local basis functions (e.g., polynomial splines $h_n(r)$ that vanish smoothly at $r_c = L/2$):

$$v_{\text{short}}(r) = \sum_{n} c_n h_n(r)$$

1. Require $h_n(r)$ to satisfy the boundary conditions ($v_{\text{short}}(r_c) = v'_{\text{short}}(r_c) = 0$).
2. Choose a desired reciprocal-space cutoff $k_c$.
3. Write down the mean-square $\chi^2$ error coming from neglecting higher wavevectors $|\mathbf{k}| > k_c$ and **minimize $\chi^2$ with respect to the expansion coefficients $c_n$**. The Natoli–Ceperley optimized breakup reduces the error $\ln[\chi/(e^2/L)]$ by several orders of magnitude compared to classic Gaussian Ewald at the same $k_c r_c$!

---

### Look at $k$-Space: Factoring the Double Sum & Avoiding Complex Exponentiation
Naively, the reciprocal-space sum $\frac{1}{2}\sum_{\mathbf{k}} \sum_{i \ne j} q_i q_j e^{i\mathbf{k}\cdot(\mathbf{r}_i - \mathbf{r}_j)} v_{\mathbf{k}}$ looks like an $\mathcal{O}(M_k N^2)$ calculation for $M_k$ wavevectors. However, we can factor the double sum over $(i, j)$ into the product of **single-particle structure factors** $\rho_{\mathbf{k}}$:

$$V_{\text{long}} = \frac{1}{2}\sum_{\mathbf{k}} \left[\sum_{i=1}^N q_i e^{i\mathbf{k}\cdot\mathbf{r}_i}\right]\left[\sum_{j=1}^N q_j e^{-i\mathbf{k}\cdot\mathbf{r}_j}\right] v_{\mathbf{k}} + C = \frac{1}{2}\sum_{\mathbf{k}} \rho_{\mathbf{k}} \rho_{-\mathbf{k}} v_{\mathbf{k}} + C$$

* **Hermitian Symmetry ($\rho_{-\mathbf{k}} = \rho_{\mathbf{k}}^*$):** Because particle positions $\mathbf{r}_i$ are real, $\rho_{-\mathbf{k}} = \rho_{\mathbf{k}}^*$, so we only need to compute half of the $\mathbf{k}$-space sphere!
* **Linear Scaling in $N$ per $\mathbf{k}$-vector ($\mathcal{O}(N M_k)$):** Computing $\rho_{\mathbf{k}} = \sum_{i=1}^N q_i e^{i\mathbf{k}\cdot\mathbf{r}_i}$ for $M_k$ wavevectors takes **$\mathcal{O}(N M_k)$** operations instead of $\mathcal{O}(N^2 M_k)$.
* **Fast Single-Particle MC Updates ($\mathcal{O}(M_k)$):** If only one or a few particles move (as in Monte Carlo), we do not recompute $\rho_{\mathbf{k}}$ from scratch! We simply update $\Delta \rho_{\mathbf{k}} = q_i (e^{i\mathbf{k}\cdot\mathbf{r}_i^{\text{new}}} - e^{i\mathbf{k}\cdot\mathbf{r}_i^{\text{old}}})$ in $\mathcal{O}(M_k)$ time.
* **Recursive Complex Multiplication Trick:** Calling `exp(i k·r)` or `sin`/`cos` from scratch for every $(i, \mathbf{k})$ pair is slow. Instead, for $\mathbf{k} = \left(m_1 \frac{2\pi}{L_x}, m_2 \frac{2\pi}{L_y}, m_3 \frac{2\pi}{L_z}\right)$, we factor:
 $$e^{i\mathbf{k}\cdot\mathbf{r}_i} = \left[e^{i\frac{2\pi}{L_x}r_{ix}}\right]^{m_1} \left[e^{i\frac{2\pi}{L_y}r_{iy}}\right]^{m_2} \left[e^{i\frac{2\pi}{L_z}r_{iz}}\right]^{m_3}$$
 For each particle $i$, we compute the fundamental base phase $C_j = \exp(i k_{0j} r_{ij}) = (\cos(k_{0j}r_{ij}), \sin(k_{0j}r_{ij}))$ only **3 times** ($\mathcal{O}(N)$ total), generate integer powers $C_{j,m} = (C_j)^m$ iteratively by fast **complex multiplication** $e^{i m k_0 x} = e^{i(m-1)k_0 x}\cdot e^{i k_0 x}$, and multiply the three Cartesian factors together!

---

### Derivation of Optimal Ewald $\mathcal{O}(N^{3/2})$ Scaling vs. PME $\mathcal{O}(N\ln N)$ vs. FMM $\mathcal{O}(N)$
At fixed liquid density $\rho = N / \Omega$ ($\Omega = L^3 \propto N$) and fixed target accuracy $\epsilon = e^{-p}$:
* **Real-space work:** Cutoff $r_c \sim \sqrt{p}/\kappa$. Using a neighbor list, each of the $N$ particles interacts with $\frac{4}{3}\pi\rho r_c^3$ neighbors:
 $$W_{\text{real}} \propto N \rho r_c^3 \propto \frac{N^2}{\Omega \kappa^3}$$
* **Reciprocal-space work:** Cutoff $k_c \sim 2\kappa\sqrt{p}$. The number of discrete $\mathbf{k}$-vectors inside radius $k_c$ is $M_k \propto \Omega k_c^3 \propto \Omega \kappa^3$. Summing $\rho_{\mathbf{k}}$ over $N$ particles for each of the $M_k$ wavevectors costs:
 $$W_{\text{recip}} \propto N M_k \propto N \Omega \kappa^3$$

#### Balancing Real and Reciprocal Work
* **Unoptimized $\kappa$ (fixing $r_c = L/2$, $\kappa \propto 1/L \propto N^{-1/3}$):** Then $W_{\text{real}} \propto \mathcal{O}(N^2)$ and $W_{\text{recip}} \propto \mathcal{O}(N)$, so unoptimized Ewald scales as **$\mathcal{O}(N^2)$**!
* **Optimized $\kappa$:** Setting $W_{\text{real}} \sim W_{\text{recip}}$ requires:
 $$\frac{N^2}{\Omega \kappa^3} \sim N \Omega \kappa^3 \implies \kappa_{\text{opt}} \propto \frac{N^{1/6}}{\Omega^{1/3}} = \frac{N^{1/6}}{L} \propto \rho^{1/3} N^{-1/6}$$
 Substituting $\Omega \kappa_{\text{opt}}^3 \propto N^{1/2}$ into both work expressions gives the celebrated **Optimal Ewald complexity**:
 $$W_{\text{real}} \sim W_{\text{recip}} \sim N \cdot N^{1/2} = \mathcal{O}(N^{3/2}), \qquad M_k \propto \mathcal{O}(N^{1/2})$$
 *( step breakdown at optimal $\kappa$: 1. Base $\exp(ik_0 x_i)$ is $\mathcal{O}(N)$; 2. Powers & products $\mathcal{O}(N^{3/2})$; 3. Sum over particles for $\rho_{\mathbf{k}}$ is $\mathcal{O}(N^{3/2})$; 4. Sum over $\mathbf{k}$ for potential is $\mathcal{O}(M_k) = \mathcal{O}(N^{1/2})$; 5. Forces via gradients $\mathcal{O}(N^{3/2})$; 6. Self/constant terms $\mathcal{O}(1)$. Code verification check: a unit-charge simple cubic lattice in a neutralizing background has Wigner energy $V = -1.4186487 / a$.)*

#### Faster Asymptotic Methods for Large $N$
1. **Particle-Mesh / Particle-Cell Methods (Hockney PPPM, Darden PME — $\mathcal{O}(N \ln N)$):**
 Instead of evaluating $\rho_{\mathbf{k}}$ by direct $\mathcal{O}(N M_k)$ summation, charges are interpolated onto a 3D regular grid and Poisson's equation is solved in $k$-space using the **3D Fast Fourier Transform (FFT)**, reducing reciprocal-space cost to **$\mathcal{O}(N \ln N)$**.
2. **Fast Multipole Method (Greengard & Rokhlin FMM — $\mathcal{O}(N)$):**
 Divides space recursively into an octree of cells, computes multipole (monopole, dipole, quadrupole, $\dots$) moments of each cell, and translates moments hierarchically between parent supercells and child cells, achieving true linear **$\mathcal{O}(N)$** complexity for very large $N$.

---

### Artifacts of Periodic Image Potentials in Inhomogeneous / Slab Systems: Ewald summation assumes the simulation cell is replicated periodically in all 3 directions to infinity. For an **inhomogeneous system** such as a **2D charged surface, membrane, or liquid–vapor interface (charge layer)**, 3D periodic images impose an artificial long-range crystal lattice of parallel dipole layers! Whereas a real nearby fluid would be **anticorrelated** (screening the local dipole), periodic images are **perfectly correlated** replicas, artificially **penalizing** localized dipole fluctuations unless a 2D slab dipole correction or vacuum gap is applied.


In [31]:
ws.render_section('concept_electrostatic_scaling')

### Interactive Explorer: Electrostatic Solver Complexity — O(N²) vs. O(N^(3/2)) vs. O(N ln N) vs. O(N)

Compare the per-step CPU time on log–log axes across five orders of magnitude in particle count $N$ ($10^2$ to $10^7$) for Direct/Unoptimized Ewald $\mathcal{O}(N^2)$, Optimized Ewald $\mathcal{O}(N^{3/2})$, Particle-Mesh Ewald (PME/PPPM) $\mathcal{O}(N\ln N)$, and the Fast Multipole Method (FMM) $\mathcal{O}(N)$.

**Key Takeaway:** Optimized Ewald ($\mathcal{O}(N^{3/2})$) has a tiny prefactor and excels for small systems ($N \lesssim 10^3$), PME/PPPM ($\mathcal{O}(N\ln N)$) dominates intermediate biomolecular and materials simulations ($10^3 \lesssim N \lesssim 10^5$), and FMM ($\mathcal{O}(N)$) wins at ultra-large scales ($N \gtrsim 10^5$–$10^6$) depending on multipole order $p$.

In [32]:
ws.render_section('q23_kspace_factorization_trick')

**Q23 · easy** · 💡

How does the Ewald algorithm avoid an $\mathcal{O}(N^2)$ double loop over particle pairs $(i, j)$ when evaluating the reciprocal-space energy $V_{\text{long}}$ for a given wavevector $\mathbf{k}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Look at k-space: the algorithm'.

</div>
</details>

- [ ] **(A)** It factors $\exp(i\mathbf{k}\cdot(\mathbf{r}_i - \mathbf{r}_j))$ into the single-particle sum $\rho_{\mathbf{k}} = \sum_i q_i \exp(i\mathbf{k}\cdot\mathbf{r}_i)$ and its complex conjugate $\rho_{-\mathbf{k}} = \rho_{\mathbf{k}}^*$, reducing the cost per $k$-vector from $\mathcal{O}(N^2)$ to $\mathcal{O}(N)$.
- [ ] **(B)** It truncates the sum over i and j after the first 10 particles in the array.
- [ ] **(C)** It sets $\exp(i\mathbf{k}\cdot(\mathbf{r}_i - \mathbf{r}_j)) = 1$ for all pairs in $k$-space.
- [ ] **(D)** It uses a Verlet skin list in k-space so each particle only interacts with 12 k-vectors.

In [33]:
ws.render_section('q24_rho_k_hermitian_symmetry')

**Q24 · easy** · 💡

Because particle coordinates $\mathbf{r}_i$ and charges $q_i$ are real numbers, how is the structure factor at $-\mathbf{k}$ related to $\rho_{\mathbf{k}} = \sum_{j=1}^N q_j e^{i\mathbf{k}\cdot\mathbf{r}_j}$, and what computational savings does this provide?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Take the complex conjugate of $\rho_{\mathbf{k}}$.

</div>
</details>

- [ ] **(A)** $\rho_{-\mathbf{k}} = \rho_{\mathbf{k}}^*$ (complex conjugate), so we only need to compute $\rho_{\mathbf{k}}$ over half of the reciprocal-space $k$-vectors.
- [ ] **(B)** $\rho_{-\mathbf{k}} = -\rho_{\mathbf{k}}$, so $\rho_{\mathbf{k}} \rho_{-\mathbf{k}}$ cancels to zero for every $\mathbf{k}$.
- [ ] **(C)** $\rho_{-\mathbf{k}} = 1 / \rho_{\mathbf{k}}$, so $\rho_{\mathbf{k}} \rho_{-\mathbf{k}} = 1$ identically.
- [ ] **(D)** $\rho_{-\mathbf{k}}$ is independent of $\rho_{\mathbf{k}}$ and requires doubling the $k$-space cutoff.

In [34]:
ws.render_section('q25_fast_complex_multiply_algorithm')

**Q25 · medium** · 💡

When computing $e^{i\mathbf{k}\cdot\mathbf{r}_i}$ across hundreds of reciprocal lattice vectors $\mathbf{k} = \left(m_1 \frac{2\pi}{L_x}, m_2 \frac{2\pi}{L_y}, m_3 \frac{2\pi}{L_z}\right)$, how do efficient Ewald codes avoid calling slow trigonometric or complex exponential library functions inside the inner loop over $\mathbf{k}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check on factoring $e^{i\mathbf{k}\cdot\mathbf{r}}$ into powers of fundamental phases $[e^{i(2\pi/L_x)r_x}]^{m_1}$.

</div>
</details>

- [ ] **(A)** They compute the base phases $\exp(i k_0 x_i) = (\cos(k_0 x_i), \sin(k_0 x_i))$ once per particle in $\mathcal{O}(N)$ time and generate all higher harmonics $m_1, m_2, m_3$ and 3D products purely by fast complex multiplications.
- [ ] **(B)** They replace $\exp(i\mathbf{k}\cdot\mathbf{r})$ with its first-order Taylor expansion $1 + i\mathbf{k}\cdot\mathbf{r}$ for all $\mathbf{k}$.
- [ ] **(C)** They round all particle coordinates $\mathbf{r}_i$ to integers so $\exp(i\mathbf{k}\cdot\mathbf{r})$ is always $\pm 1$.
- [ ] **(D)** They evaluate $\exp(i\mathbf{k}\cdot\mathbf{r})$ using a 100-term series of Bessel functions at each $k$-vector.

In [35]:
ws.render_section('q26_single_particle_mc_ewald_update')

**Q26 · medium** · 💡

In a Monte Carlo simulation of $N$ ions using Ewald summation with $M_k$ reciprocal-space wavevectors, suppose a trial move displaces only **a single ion $i$** from $\mathbf{r}_i^{\text{old}}$ to $\mathbf{r}_i^{\text{new}}$. How many operations are needed to update the reciprocal-space energy $V_{\text{long}}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the bottom of: 'Changes due to moving a few particles can be calculated more quickly'.

</div>
</details>

- [ ] **(A)** $\mathcal{O}(M_k)$ operations (or $\mathcal{O}(N^{1/2})$ at optimal $\kappa$), by updating $\rho_{\mathbf{k}}^{\text{new}} = \rho_{\mathbf{k}}^{\text{old}} + q_i [\exp(i\mathbf{k}\cdot\mathbf{r}_i^{\text{new}}) - \exp(i\mathbf{k}\cdot\mathbf{r}_i^{\text{old}})]$ for each $k$-vector without re-summing the other $N - 1$ particles.
- [ ] **(B)** $\mathcal{O}(N^2 M_k)$ operations, because every pair distance $r_{jk}$ must be Fourier transformed from scratch.
- [ ] **(C)** $\mathcal{O}(N^3)$ operations via Cholesky decomposition of the $k$-space matrix.
- [ ] **(D)** Zero operations, because $V_{\text{long}}$ does not depend on individual particle positions.

In [36]:
ws.render_section('q27_natoli_ceperley_optimized_breakup')

**Q27 · medium** · 💡

How does the **Natoli–Ceperley (1995) optimized breakup** improve upon the conventional Gaussian Ewald splitting?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Ewald: optimizing breakup'.

</div>
</details>

- [ ] **(A)** It expands $v_{\text{short}}(r) = \sum_n c_n h_n(r)$ in basis functions satisfying boundary conditions at $r_c = L/2$ and optimizes the coefficients $c_n$ to minimize the $k$-space truncation error from neglecting $|\mathbf{k}| > k_c$.
- [ ] **(B)** It replaces the Coulomb potential $1/r$ with a hard-sphere potential at all distances.
- [ ] **(C)** It sets the $k$-space cutoff $k_c = 0$ so that no Fourier vectors are ever needed.
- [ ] **(D)** It uses a wider Gaussian so that $v_{\text{short}}(L/2)$ is maximized at the box boundary.

In [37]:
ws.render_section('q28_optimal_ewald_complexity_derivation')

**Q28 · medium** · 💡

When simulating $N$ charges at constant density ($\Omega \propto N$), what is the per-step computational complexity of the Ewald sum if (i) $\kappa$ is **not** re-optimized with $N$ ($r_c = L/2$ fixed fraction of box) versus (ii) $\kappa$ **is** optimally tuned ($\kappa \propto N^{1/6}/L$) together with neighbor lists?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Use neighbor tables and optimize κ, Ewald is O(N^(3/2)); If we do not reoptimize, then O(N²)'.

</div>
</details>

- [ ] **(A)** $\mathcal{O}(N^2)$ if $\kappa$ is not re-optimized; $\mathcal{O}(N^{3/2})$ when $\kappa$ is optimally tuned and neighbor lists are used.
- [ ] **(B)** $\mathcal{O}(N^3)$ if $\kappa$ is not re-optimized; $\mathcal{O}(N^2)$ when $\kappa$ is optimally tuned.
- [ ] **(C)** $\mathcal{O}(N\ln N)$ in both cases.
- [ ] **(D)** $\mathcal{O}(N)$ if $\kappa$ is not re-optimized; $\mathcal{O}(N^{3/2})$ when $\kappa$ is optimally tuned.

In [38]:
ws.render_section('q29_ewald_scaling_factor_numeric')

**Q29 · easy** · 💡

In an optimally tuned Ewald summation code whose CPU time per step scales as $\mathcal{O}(N^{3/2})$, by what factor does the CPU time per step increase if the number of ions $N$ is increased by a factor of $4$ (e.g., from $N = 1{,}000$ to $N = 4{,}000$ at constant density)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Raise the system size scaling ratio to the $3/2$ power.

</div>
</details>

_Numeric answer required._

In [39]:
ws.render_section('q30_ewald_step_by_step_complexities')

**Q30 · hard** · 💡

In an optimally tuned Ewald sum ($\mathcal{O}(N^{3/2})$ overall, where the number of $\mathbf{k}$-vectors scales as $M_k \propto N^{1/2}$), what are the individual asymptotic scalings of: (1) computing the fundamental 1D phases $\exp(ik_0 x_i)$ for all particles, (3) summing over all $N$ particles to get $\rho_{\mathbf{k}}$ for all $M_k$ wavevectors, and (4) summing $|\rho_{\mathbf{k}}|^2 v_{\mathbf{k}}$ over the $M_k$ wavevectors to get the total reciprocal-space potential?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the right-hand column of ('How to do it').

</div>
</details>

- [ ] **(A)** Step (1) is $\mathcal{O}(N)$; Step (3) is $\mathcal{O}(N^{3/2})$; Step (4) is $\mathcal{O}(N^{1/2})$.
- [ ] **(B)** Step (1) is $\mathcal{O}(N^{3/2})$; Step (3) is $\mathcal{O}(N^2)$; Step (4) is $\mathcal{O}(N)$.
- [ ] **(C)** Step (1) is $\mathcal{O}(1)$; Step (3) is $\mathcal{O}(N)$; Step (4) is $\mathcal{O}(N^{3/2})$.
- [ ] **(D)** All three steps scale as $\mathcal{O}(N^{3/2})$.

In [40]:
ws.render_section('q31_pme_and_fmm_complexities')

**Q31 · easy** · 💡

What are the asymptotic CPU complexities for large $N$ of (i) **Particle-Cell / Particle-Mesh Ewald methods (Hockney PPPM / PME)** that use Fast Fourier Transforms on a grid, and (ii) the **Fast Multipole Method (FMM)** that recursively divides space into cells and translates multipole moments?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Complexity of Fast Multipole'.

</div>
</details>

- [ ] **(A)** Particle-Mesh / FFT methods scale as $\mathcal{O}(N\ln N)$, and Fast Multipole Methods (FMM) scale as $\mathcal{O}(N)$.
- [ ] **(B)** Particle-Mesh / FFT methods scale as $\mathcal{O}(N^2)$, and Fast Multipole Methods scale as $\mathcal{O}(N^{3/2})$.
- [ ] **(C)** Both Particle-Mesh and Fast Multipole Methods scale as $\mathcal{O}(N^{1/2})$.
- [ ] **(D)** Particle-Mesh scales as $\mathcal{O}(N)$, and Fast Multipole scales as $\mathcal{O}(N^3)$.

In [41]:
ws.render_section('q32_image_potential_slab_artifact')

**Q32 · medium** · 💡

what physical artifact arises when standard 3D periodic Ewald summation is applied to an inhomogeneous **charge layer or dipolar surface**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare how a real surrounding fluid responds to a local dipole/charge layer versus what periodic replicas do.

</div>
</details>

- [ ] **(A)** 3D periodic images force the charge/dipole structure to repeat identically at large r (correlated replicas), whereas a real nearby fluid would be anticorrelated (screening); thus such structures are artificially penalized.
- [ ] **(B)** The Ewald sum sets all forces perpendicular to the layer to zero.
- [ ] **(C)** The self-interaction term $V_{\text{self}}$ becomes infinite whenever a surface is present.
- [ ] **(D)** Periodic boundary conditions cause all ions to evaporate into vacuum immediately.

In [42]:
ws.render_section('q33_verlet_neighbor_count_numeric')

**Q33 · medium** · 💡

A 3D liquid has number density $\rho = 0.75\,\sigma^{-3}$. If a Verlet neighbor list is constructed with potential cutoff $r_c = 2.5\,\sigma$ and skin depth $\Delta = 0.5\,\sigma$ (so $r_{\text{list}} = r_c + \Delta = 3.0\,\sigma$), what is the average number of neighbors $m$ stored per particle (assuming $g(r) \approx 1$, round to 1 decimal place)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate the expected number of particles inside a sphere of radius $r_c + \Delta$ at the given number density.

</div>
</details>

_Numeric answer required._

In [43]:
ws.render_section('q34_cell_method_3d_neighbor_cells_numeric')

**Q34 · easy** · 💡

In the 3D linked-cell method, the simulation box is divided into cubic cells of side length $L_{\text{cell}} > r_c + \Delta$. To find all neighbors of a particle within $r_c + \Delta$ (without using Newton's 3rd law half-shell optimization), how many total cells must be searched for that particle (counting the particle's own cell plus all immediately adjacent cells in 3D)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider a $3 \times 3 \times 3$ cube of cells centered on the particle's cell.

</div>
</details>

_Numeric answer required._

In [44]:
ws.render_section('q35_ewald_real_space_erfc_cutoff_numeric')

**Q35 · medium** · 💡

In classic Ewald summation in a cubic box of length $L = 10.0\text{ Å}$, we choose $\kappa$ so that the real-space screened potential $v_{\text{short}}(r) = \frac{q_1 q_2}{r}\text{erfc}(\kappa r)$ is negligible at the minimum-image boundary $r = L/2 = 5.0\text{ Å}$. Using the asymptotic approximation $\text{erfc}(z) \approx \frac{e^{-z^2}}{z\sqrt{\pi}}$ for large $z$, estimate $\kappa$ (in $\text{Å}^{-1}$) required so that $z = \kappa(L/2) = 3.0$ (where $\text{erfc}(3.0) \approx 2.2 \times 10^{-5}$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Divide the target dimensionless argument $z = 3.0$ by the minimum-image cutoff distance $L/2$.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q36_one_component_plasma_background')

**Q36 · easy** · 💡

If you want to simulate a **one-component charged system** (such as an electron gas or a lattice of $+1$ ions with no counter-ions) in periodic boundary conditions, what must be added to prevent the electrostatic energy ($\mathbf{k} = \mathbf{0}$ term) from diverging to infinity?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check and ('For one component system, need to add a background to make it neutral').

</div>
</details>

- [ ] **(A)** A uniform neutralizing background charge density of opposite sign (jellium background), which exactly cancels the $k = 0$ divergence.
- [ ] **(B)** A global Langevin friction term with $\zeta \to \infty$.
- [ ] **(C)** An extra repulsive $1/r^{12}$ Lennard-Jones core at $r = 0$.
- [ ] **(D)** Nothing; the sum $\sum_{\mathbf{L}\ne 0} 1/|\mathbf{r} - \mathbf{L}|$ converges to zero on its own.

In [46]:
ws.render_section('q37_cubic_lattice_wigner_check')

**Q37 · easy** · 💡

what benchmark test is recommended to verify that a newly written 3D Ewald summation code (including real-space, $k$-space, and constant self/background terms) is bug-free and independent of $\kappa$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the last bullet of ('Checks: perfect cubic lattice...').

</div>
</details>

- [ ] **(A)** Compute the potential energy of a perfect simple cubic lattice of unit charges in a neutralizing background and verify that $V = -1.4186487 / a$ (independent of the choice of $\kappa$).
- [ ] **(B)** Verify that the total electrostatic energy is proportional to $\kappa^3$ when $\kappa$ is varied.
- [ ] **(C)** Check that setting all charges $q_i = +1$ gives $V = +\infty$.
- [ ] **(D)** Verify that the real-space term alone equals zero for any $\kappa$.

In [47]:
ws.render_section('q38_ewald_independence_of_kappa')

**Q38 · medium** · 💡

When both the real-space sum ($r < r_c$) and reciprocal-space sum ($|\mathbf{k}| < k_c$) are converged to high accuracy, how should the **total** Ewald potential energy $V_{\text{Ewald}} = V_{\text{real}}(\kappa) + V_{\text{recip}}(\kappa) + V_{\text{self}}(\kappa) + V_{\text{dipole}}$ depend on the screening parameter $\kappa$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Remember that $\kappa$ only controls how we split $\frac{1}{r} = \frac{\text{erfc}(\kappa r)}{r} + \frac{\text{erf}(\kappa r)}{r}$.

</div>
</details>

- [ ] **(A)** $V_{\text{Ewald}}$ is completely independent of $\kappa$ ($\partial V_{\text{Ewald}} / \partial\kappa = 0$), even though $V_{\text{real}}$, $V_{\text{recip}}$, and $V_{\text{self}}$ each vary strongly with $\kappa$.
- [ ] **(B)** $V_{\text{Ewald}}$ increases linearly with $\kappa$ because $V_{\text{self}} \propto -\kappa$.
- [ ] **(C)** $V_{\text{Ewald}}$ is proportional to $1/\kappa^2$.
- [ ] **(D)** $V_{\text{Ewald}}$ oscillates sinusoidally with $\kappa$.

In [48]:
ws.render_section('q39_fast_multipole_hierarchical_steps')

**Q39 · medium** · 💡

Which sequence of steps describes how the **Fast Multipole Method (FMM)** achieves $\mathcal{O}(N)$ complexity for long-range Coulomb interactions?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the bullet points under 'Fast Multipole Methods are O(N) for large N'.

</div>
</details>

- [ ] **(A)** Divide space into cells recursively, compute the multipole (dipole, quadrupole,...) moments of each small cell, and apply translation rules relating supercell moments to smaller cell moments.
- [ ] **(B)** Truncate the Coulomb potential at nearest neighbors and add a constant density tail correction.
- [ ] **(C)** Diagonalize the 3N × 3N Oseen matrix via Cholesky decomposition at every timestep.
- [ ] **(D)** Transform all coordinates into 1D chains and multiply by 2 ln 2.

In [49]:
ws.render_section('q40_lj_total_tail_correction_zero_crossing_numeric')

**Q40 · hard** · 💡

For the full 12-6 Lennard-Jones potential $V_{\text{LJ}}(r) = 4\varepsilon[(\sigma/r)^{12} - (\sigma/r)^6]$ truncated at $r_c = 2.0\,\sigma$ in a 3D fluid of reduced density $\rho\sigma^3 = 0.75$ and $\varepsilon = 1.0$, calculate the complete per-particle energy tail correction:

$$\frac{\Delta E_{\text{tail}}^{\text{LJ}}}{N} = \frac{8\pi\rho\varepsilon\sigma^3}{3}\left[ \frac{1}{3}\left(\frac{\sigma}{r_c}\right)^9 - \left(\frac{\sigma}{r_c}\right)^3 \right]$$

(Round to 3 decimal places; notice how small the $r^{-12}$ repulsive tail is compared to the $r^{-6}$ attractive tail at $r_c = 2\sigma$!)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute the density and cutoff ratio into the integrated Lennard-Jones tail formula.

</div>
</details>

_Numeric answer required._

In [50]:
ws.render_section('q41_optimal_kspace_vectors_scaling_numeric')

**Q41 · medium** · 💡

In an optimally tuned Ewald simulation ($\kappa \propto N^{1/6}/L$), the number of reciprocal-space wavevectors $M_k$ inside the cutoff sphere $|\mathbf{k}| < k_c$ scales as $M_k \propto \Omega \kappa^3 \propto N^{1/2}$. If an optimal Ewald simulation of $N_1 = 500$ ions uses $M_{k,1} = 120$ wavevectors, how many wavevectors $M_{k,2}$ should be used for a system $9$ times larger ($N_2 = 4{,}500$ ions at the same density) to maintain optimal $\mathcal{O}(N^{3/2})$ balance?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Scale the initial wavevector count by the square root of the system size expansion ratio.

</div>
</details>

_Numeric answer required._

In [51]:
ws.render_section('q42_long_range_and_ewald_synthesis_multiselect')

**Q42 · hard** · 💡

Which of the following statements about neighbor lists, long-range potentials, and Ewald summation are true? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check each statement against the 3D long-range criterion, Stillinger–Lovett screening, Ewald k-space factorization, and linked-cell scaling.

</div>
</details>

- [ ] **(A)** Combining a linked-cell mesh (cell size $L_{\text{cell}} > r_c + \Delta$) with a Verlet skin list reduces both force evaluation and neighbor-table construction to $\mathcal{O}(N)$.
- [ ] **(B)** In 3D, any potential decaying as $r^{-\nu}$ with $\nu \le 3$ (including Coulomb $1/r$ and dipole $1/r^3$) is long-ranged because $\int r^2 r^{-\nu}\,dr$ diverges.
- [ ] **(C)** By factoring $\exp(i\mathbf{k}\cdot(\mathbf{r}_i - \mathbf{r}_j))$ into single-particle sums $\rho_{\mathbf{k}} \rho_{-\mathbf{k}}$ and tuning $\kappa \propto N^{1/6}/L$, classic Ewald summation achieves $\mathcal{O}(N^{3/2})$ complexity.
- [ ] **(D)** Truncating a 3D Coulomb potential sharply at $r_c = L/2$ satisfies Poisson's equation and preserves the Stillinger–Lovett condition $S(k) = c k^2$.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()